# Week 8: Multi-Agent Freelancer Opportunity Hunter

This project implements a **multi-agent system** that discovers and evaluates freelance job opportunities using:

- Multi-agent orchestration
- Semantic search with embeddings
- Vector memory using ChromaDB
- Automated proposal generation
- Persistent memory

---

## 🤖 Agents

- Scanner Agent → Fetch jobs
- Extractor Agent → Clean & structure data
- Deduplication Agent → Remove similar jobs (vector similarity)
- Fit Scoring Agent → Semantic matching (embeddings)
- Ranking Agent → Rank best opportunities
- Proposal Agent → Generate tailored proposals
- Planner Agent → Orchestrates everything
- Notifier Agent → Logs results

---

## 🧠 Key Features

- ✅ Semantic matching (NOT keyword-based)
- ✅ Vector database (ChromaDB)
- ✅ Memory + deduplication
- ✅ Multi-agent collaboration
- ✅ Extendable to real APIs

---

> This is a demo system with mock data. Easily extendable to real platforms like Upwork or LinkedIn.

In [1]:
!pip install -q gradio chromadb sentence-transformers pydantic plotly numpy

zsh:1: command not found: pip


In [ ]:
import json
import time
from typing import List
import numpy as np

import gradio as gr
import chromadb
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel

In [3]:
class JobRecord(BaseModel):
    title: str
    description: str
    budget: float
    skills: List[str]
    link: str


class JobOpportunity(BaseModel):
    job: JobRecord
    score: float
    proposal: str

In [4]:
class BaseAgent:
    label = "Agent"

    def log(self, msg):
        print(f"[{self.label}] {msg}")

In [18]:
class EmbeddingService:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def embed(self, texts):
        return self.model.encode(texts).tolist()


class VectorStore:
    def __init__(self):
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection("jobs")
        self.embedder = EmbeddingService()

    def add_jobs(self, jobs: List[JobRecord]):
        docs = [job.description for job in jobs]
        embeddings = self.embedder.embed(docs)

        # sanitize metadata
        metadatas = []
        for job in jobs:
            data = job.model_dump()

            # convert list → string (IMPORTANT)
            if isinstance(data.get("skills"), list):
                data["skills"] = ", ".join(data["skills"])

            metadatas.append(data)

        self.collection.add(
            documents=docs,
            embeddings=embeddings,
            ids=[job.link for job in jobs],
            metadatas=metadatas
        )

    def search(self, query: str, n=3):
        emb = self.embedder.embed([query])[0]

        return self.collection.query(
            query_embeddings=[emb],
            n_results=n
        )

In [19]:
class JobScannerAgent(BaseAgent):
    label = "Scanner"

    def scan(self):
        self.log("Scanning jobs...")
        time.sleep(1)

        return [
            JobRecord(
                title="Senior Next.js Developer",
                description="Looking for Next.js expert with Node.js and scalable architecture experience.",
                budget=1200,
                skills=["Next.js", "Node.js"],
                link="job1"
            ),
            JobRecord(
                title="NestJS API Developer",
                description="Build scalable backend using NestJS and MongoDB.",
                budget=900,
                skills=["NestJS", "MongoDB"],
                link="job2"
            )
        ]

In [20]:
class ExtractorAgent(BaseAgent):
    label = "Extractor"

    def extract(self, jobs):
        self.log("Structuring jobs...")
        return jobs

In [21]:
class DeduplicationAgent(BaseAgent):
    label = "Deduplicator"

    def __init__(self, vector_store: VectorStore):
        self.vector_store = vector_store

    def filter(self, jobs):
        self.log("Removing duplicates (semantic)...")

        new_jobs = []

        for job in jobs:
            results = self.vector_store.search(job.description, n=1)

            if results["distances"] and len(results["distances"][0]) > 0:
                distance = results["distances"][0][0]

                # lower distance = more similar
                if distance < 0.2:
                    continue

            new_jobs.append(job)

        return new_jobs

In [22]:
class FitScoringAgent(BaseAgent):
    label = "Scorer"

    def __init__(self, vector_store: VectorStore):
        self.vector_store = vector_store
        self.profile = "Senior fullstack dev: Next.js, Node.js, NestJS, MongoDB"

    def score(self, jobs):
        self.log("Scoring jobs...")

        profile_emb = self.vector_store.embedder.embed([self.profile])[0]

        results = []

        for job in jobs:
            job_emb = self.vector_store.embedder.embed([job.description])[0]

            sim = np.dot(profile_emb, job_emb) / (
                np.linalg.norm(profile_emb) * np.linalg.norm(job_emb)
            )

            results.append((job, sim * 100))

        return results

In [23]:
class RankingAgent(BaseAgent):
    label = "Ranker"

    def rank(self, scored):
        self.log("Ranking jobs...")
        return sorted(scored, key=lambda x: x[1], reverse=True)

In [24]:
class ProposalAgent(BaseAgent):
    label = "Proposal"

    def generate(self, job):
        self.log(f"Generating proposal for {job.title}")

        return f"""
Hi,

I have strong experience in {', '.join(job.skills)} and building scalable systems.

I can deliver high-quality results for your project.

Let's discuss!

Best,
"""

In [25]:
class NotifierAgent(BaseAgent):
    label = "Notifier"

    def notify(self, opp):
        self.log(f"Top Job: {opp.job.title} | Score: {opp.score:.2f}")

In [26]:
class PlannerAgent(BaseAgent):
    label = "Planner"

    def __init__(self):
        self.vector_store = VectorStore()

        self.scanner = JobScannerAgent()
        self.extractor = ExtractorAgent()
        self.dedup = DeduplicationAgent(self.vector_store)
        self.scorer = FitScoringAgent(self.vector_store)
        self.ranker = RankingAgent()
        self.proposal = ProposalAgent()
        self.notifier = NotifierAgent()

    def run(self):
        jobs = self.scanner.scan()
        jobs = self.extractor.extract(jobs)
        jobs = self.dedup.filter(jobs)

        if not jobs:
            self.log("No new jobs found.")
            return []

        self.vector_store.add_jobs(jobs)

        scored = self.scorer.score(jobs)
        ranked = self.ranker.rank(scored)

        opportunities = []

        for job, score in ranked:
            proposal = self.proposal.generate(job)

            opportunities.append(
                JobOpportunity(job=job, score=score, proposal=proposal)
            )

        if opportunities:
            self.notifier.notify(opportunities[0])

        return opportunities

In [27]:
class MemoryStore:
    FILE = "memory.json"

    def save(self, data):
        with open(self.FILE, "w") as f:
            json.dump(data, f, indent=2)

In [28]:
planner = PlannerAgent()
memory = MemoryStore()

def run_pipeline():
    results = planner.run()

    memory.save([r.dict() for r in results])

    table = []
    for r in results:
        table.append([
            r.job.title,
            r.job.budget,
            round(r.score, 2),
            r.job.link
        ])

    return table

In [29]:
with gr.Blocks() as app:
    gr.Markdown("# 🤖 Freelancer Opportunity Hunter")

    table = gr.Dataframe(headers=["Title", "Budget", "Score", "Link"])

    btn = gr.Button("Scan Jobs")
    btn.click(run_pipeline, outputs=table)

app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[Scanner] Scanning jobs...
[Extractor] Structuring jobs...
[Deduplicator] Removing duplicates (semantic)...
[Scorer] Scoring jobs...
[Ranker] Ranking jobs...
[Proposal] Generating proposal for NestJS API Developer
[Proposal] Generating proposal for Senior Next.js Developer
[Notifier] Top Job: NestJS API Developer | Score: 67.53


/var/folders/13/c33dyrs50qsfzm5d1gcrplvr0000gn/T/ipykernel_8101/192392622.py:7: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  memory.save([r.dict() for r in results])
